# parity

> Compare the MAX graph with Hugging Face on a tiny wav2vec 2.0 of the same weights.
> 
> The waveform normalization is checked first. The model is then compiled
> once and compared on CTC logits and final hidden states. If
> `models/wav2vec2.safetensors` exists, the real checkpoint is compared too.


In [ ]:
#| default_exp parity


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np

from nada.config import base_960h_config, tiny_config
from nada.features import normalize, prepare
from nada.model import build, compile_model, load_checkpoint, to_device
from nada.weights import assign_parameters, fold_weight_norm, usable


In [ ]:
#| export
def main() -> None:
    check_normalize()
    check_model()
    check_pretrained()
    print("parity ok")


In [ ]:
#| export
def check_normalize() -> None:
    from transformers import Wav2Vec2FeatureExtractor

    extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16_000, do_normalize=True)
    clip = np.random.default_rng(0).standard_normal(1600) * 0.1 + 0.05
    reference = extractor(clip, sampling_rate=16_000, return_tensors="np")["input_values"]
    np.testing.assert_allclose(normalize(clip)[None], reference, atol=1e-5, rtol=1e-5)
    print("normalize ok", float(np.max(np.abs(normalize(clip)[None] - reference))))


In [ ]:
#| export
def _tiny_reference():
    import torch
    from transformers import Wav2Vec2Config, Wav2Vec2ForCTC

    config = tiny_config()
    reference = Wav2Vec2ForCTC(
        Wav2Vec2Config(
            vocab_size=config.vocab_size,
            hidden_size=config.hidden_size,
            num_hidden_layers=config.num_hidden_layers,
            num_attention_heads=config.num_attention_heads,
            intermediate_size=config.intermediate_size,
            layer_norm_eps=config.layer_norm_eps,
            conv_dim=config.conv_dim,
            conv_kernel=config.conv_kernel,
            conv_stride=config.conv_stride,
            conv_bias=config.conv_bias,
            num_conv_pos_embeddings=config.num_conv_pos_embeddings,
            num_conv_pos_embedding_groups=config.num_conv_pos_embedding_groups,
            feat_extract_norm="group",
            do_stable_layer_norm=False,
            apply_spec_augment=False,
            pad_token_id=config.pad_token_id,
        )
    )
    # A fresh init leaves most weights near zero. Perturb them so every path matters.
    generator = torch.Generator().manual_seed(0)
    with torch.no_grad():
        for parameter in reference.parameters():
            parameter.add_(0.2 * torch.randn(parameter.shape, generator=generator))
    return reference.eval()


In [ ]:
#| export
def _compare(name: str, got, expected, atol: float) -> None:
    got = np.asarray(got)
    delta = float(np.max(np.abs(got - expected)))
    print(f"{name} max abs {delta:.3e}")
    np.testing.assert_allclose(got, expected, atol=atol, rtol=atol)


In [ ]:
#| export
def check_model() -> None:
    import torch

    config = tiny_config()
    reference = _tiny_reference()
    model = build(config)
    arrays = {key: value.detach().cpu().numpy() for key, value in reference.state_dict().items() if usable(key)}
    assign_parameters(model, fold_weight_norm(arrays))

    waveform = np.random.default_rng(1).standard_normal((2, 400)).astype(np.float32)
    with torch.no_grad():
        output = reference(torch.from_numpy(waveform), output_hidden_states=True)
    compiled = compile_model(model, batch=2, num_samples=400)
    logits, hidden = compiled(to_device(model, waveform))
    _compare("logits", logits.to_numpy(), output.logits.numpy(), 1e-4)
    _compare("hidden", hidden.to_numpy(), output.hidden_states[-1].numpy(), 1e-4)


In [ ]:
#| export
def check_pretrained() -> None:
    import torch
    from transformers import Wav2Vec2ForCTC

    from .core import repo_root

    checkpoint = repo_root() / "models" / "wav2vec2.safetensors"
    if not checkpoint.is_file():
        print("pretrained skipped: run `pixi run convert` to compare the real checkpoint")
        return
    reference = Wav2Vec2ForCTC.from_pretrained(repo_root() / "models" / "wav2vec2-base-960h").eval()
    model = build(base_960h_config())
    load_checkpoint(model, checkpoint)

    time = np.arange(16_000, dtype=np.float64) / 16_000
    clip = (0.3 * np.sin(2 * np.pi * 220 * time) + 0.05 * np.random.default_rng(2).standard_normal(16_000)).astype(np.float32)
    values = prepare([clip])
    with torch.no_grad():
        expected = reference(torch.from_numpy(values)).logits.numpy()
    compiled = compile_model(model, batch=1, num_samples=values.shape[1])
    logits, _ = compiled(to_device(model, values))
    _compare("pretrained logits", logits.to_numpy(), expected, 5e-3)
    np.testing.assert_array_equal(np.asarray(logits.to_numpy()).argmax(-1), expected.argmax(-1))


In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    main()
